In [0]:
%pip install pendulum

In [0]:
import json
from pathlib import Path
import pendulum
import requests

In [0]:
# Detecta el catálogo actual (suele ser 'workspace' o 'main')
current_catalog = spark.sql("select current_catalog()").first()[0]
catalog = current_catalog
schema = "weather_raw"
volume = "weather"

print(f"CATALOG: {catalog}")
print(f"SCHEMA: {schema}")
print(f"VOLUME: {volume}")

In [0]:
# Crea esquema weather_raw y volumen weather si no existen
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

In [0]:
# Elimina todos los archivos y subcarpetas dentro de la ruta
dbutils.fs.rm("/Volumes/workspace/weather_raw/weather", recurse=True)

In [0]:
# Creamos los widgets
dbutils.widgets.text("start_date", "2025-01-01", "Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "2025-12-31", "End Date (YYYY-MM-DD)")
dbutils.widgets.text("output_dir", "/Volumes/workspace/weather_raw/weather", "Output Directory")
dbutils.widgets.text("api_enpoint", "https://archive-api.open-meteo.com/v1/archive", "API endpoint")
dbutils.widgets.text("daily", "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max", "Get Fields")
dbutils.widgets.text("timezone", "America/Lima", "Timezone")
dbutils.widgets.text("timeout", "30", "Time Out")

In [0]:
# Leer valores de los widgets
start_date = pendulum.parse(dbutils.widgets.get("start_date")).date()
end_date = pendulum.parse(dbutils.widgets.get("end_date")).date()
output_dir = Path(dbutils.widgets.get("output_dir"))
daily = dbutils.widgets.get("daily")
timezone = dbutils.widgets.get("timezone")
endpoint_dir = dbutils.widgets.get("api_enpoint")
timeout = dbutils.widgets.get("timeout")
API_URL = endpoint_dir
timeout = int(dbutils.widgets.get("timeout"))

<!-- Iniciamos la ingesta -->

In [0]:
# creamos una lista de con los departamentos y sus coordenadas
departamentos = [
    {"departamento": "Amazonas", "lat": -6.2317, "lon": -77.8690},
    {"departamento": "Ancash", "lat": -9.5278, "lon": -77.5278},
    {"departamento": "Apurimac", "lat": -13.6339, "lon": -72.8814},
    {"departamento": "Arequipa", "lat": -16.4090, "lon": -71.5375},
    {"departamento": "Ayacucho", "lat": -13.1588, "lon": -74.2239},
    {"departamento": "Cajamarca", "lat": -7.1617, "lon": -78.5128},
    {"departamento": "Callao", "lat": -12.0566, "lon": -77.1181},
    {"departamento": "Cusco", "lat": -13.5319, "lon": -71.9675},
    {"departamento": "Huancavelica", "lat": -12.7862, "lon": -74.9764},
    {"departamento": "Huanuco", "lat": -9.9306, "lon": -76.2422},
    {"departamento": "Ica", "lat": -14.0678, "lon": -75.7286},
    {"departamento": "Junin", "lat": -11.1589, "lon": -75.9929},
    {"departamento": "La_Libertad", "lat": -8.1116, "lon": -79.0288},
    {"departamento": "Lambayeque", "lat": -6.7714, "lon": -79.8409},
    {"departamento": "Lima", "lat": -12.0464, "lon": -77.0428},
    {"departamento": "Loreto", "lat": -3.7437, "lon": -73.2516},
    {"departamento": "Madre_de_Dios", "lat": -12.5933, "lon": -69.1891},
    {"departamento": "Moquegua", "lat": -17.1936, "lon": -70.9358},
    {"departamento": "Pasco", "lat": -10.6820, "lon": -76.2561},
    {"departamento": "Piura", "lat": -5.1945, "lon": -80.6328},
    {"departamento": "Puno", "lat": -15.8402, "lon": -70.0219},
    {"departamento": "San_Martin", "lat": -6.0342, "lon": -76.9730},
    {"departamento": "Tacna", "lat": -18.0146, "lon": -70.2536},
    {"departamento": "Tumbes", "lat": -3.5669, "lon": -80.4515},
    {"departamento": "Ucayali", "lat": -8.3791, "lon": -74.5539}
]

In [0]:
# mostramos la lista
for departamento in departamentos:
    print(f"Departamento: {departamento['departamento']}")

In [0]:

# import uuid
# Realizamos la extraccion de datos de la web de clima https://archive-api.open-meteo.com
# lectura de datos de clima del anio 2025
session = requests.Session()
for departamento in departamentos:
    saved_files = []
    current_date = start_date
    date_str = current_date.to_date_string()
    daily_output_dir = (
        output_dir
        / f"{departamento['departamento']}"
        / f"{current_date.year:04d}"
    )
    daily_output_dir.mkdir(parents=True, exist_ok=True)

    file_path = daily_output_dir / "weather.json"
    try:
        response = session.get(API_URL,  params = {
        "latitude": departamento['lat'],
        "longitude": departamento['lon'],
        "start_date": start_date,
        "end_date": end_date,
        "daily": daily,
        "timezone": timezone
    }, timeout=timeout)
        response.raise_for_status()
    except requests.RequestException:
        current_date = current_date.add(days=1)
        continue

    # Más rápido: guardar bytes del JSON tal cual lo devuelve la API
    file_path.write_bytes(response.content)
    # guardamos los archivos creados, para saer cuantos archivos se crearon por departamento
    saved_files.append(str(file_path))
    current_date = current_date.add(days=1)

    print(f"{departamento['departamento']} - Archivos guardados: {len(saved_files)}")